In [1]:
import polars as pl
import pandas as pd
import pyranges as pr

/opt/modules/i12g/anaconda/envs/sl-ukg/lib/python3.12/site-packages/sorted_nearest/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Parse GTF

In [2]:
gtf_path = "/s/genomes/Gencode/Gencode_human/release_40/gencode.v40.annotation.gtf.gz"

gtf_pl = pl.read_csv(
    gtf_path,
    separator="\t",
    comment_prefix="#",
    has_header=False,
    new_columns=["Chromosome", "source", "Feature", "Start", "End", "score", "Strand", "frame", "attributes"]
)

gtf_pl = gtf_pl.with_row_index("row_nr")
gtf_pl

row_nr,Chromosome,source,Feature,Start,End,score,Strand,frame,attributes
u64,str,str,str,i64,i64,str,str,str,str
0,"""chr1""","""HAVANA""","""exon""",11869,12227,""".""","""+""",""".""","""gene_id ""ENSG00000223972.5""; t…"
1,"""chr1""","""HAVANA""","""gene""",11869,14409,""".""","""+""",""".""","""gene_id ""ENSG00000223972.5""; g…"
2,"""chr1""","""HAVANA""","""transcript""",11869,14409,""".""","""+""",""".""","""gene_id ""ENSG00000223972.5""; t…"
3,"""chr1""","""HAVANA""","""exon""",12010,12057,""".""","""+""",""".""","""gene_id ""ENSG00000223972.5""; t…"
4,"""chr1""","""HAVANA""","""transcript""",12010,13670,""".""","""+""",""".""","""gene_id ""ENSG00000223972.5""; t…"
…,…,…,…,…,…,…,…,…,…
3283857,"""chrY""","""HAVANA""","""transcript""",57212184,57214397,""".""","""-""",""".""","""gene_id ""ENSG00000227159.8_PAR…"
3283858,"""chrY""","""HAVANA""","""exon""",57213204,57213357,""".""","""-""",""".""","""gene_id ""ENSG00000227159.8_PAR…"
3283859,"""chrY""","""HAVANA""","""exon""",57213526,57213602,""".""","""-""",""".""","""gene_id ""ENSG00000227159.8_PAR…"


In [3]:
atts = (
    gtf_pl.select("row_nr", "attributes")
    
    .with_columns(
        attrs_list=pl.col("attributes").str.split("; ")
    )
    .explode("attrs_list")
    
    .with_columns(
        pl.col("attrs_list")
        .str.split_exact(" ", 1)
        .struct.rename_fields(["attribute", "value"])
        .alias("fields")
    ).unnest("fields")

    .with_columns(
        pl.col("value").str.strip_chars('"')
    )

    .pivot(
        index="row_nr",
        on="attribute",
        values="value",
        aggregate_function=pl.element().implode()
    )

    .with_columns(
        pl.col(pl.List).list.join(", ").str.strip_chars(";").replace("", None)
    )

    .with_columns(
        level = pl.col('level').cast(pl.Int32),
        exon_number = pl.col('exon_number').cast(pl.Int32),
    )
)

atts

row_nr,gene_id,transcript_id,gene_type,gene_name,transcript_type,transcript_name,exon_number,exon_id,level,transcript_support_level,hgnc_id,tag,havana_gene,havana_transcript,ont,protein_id,ccdsid
u64,str,str,str,str,str,str,i32,str,i32,str,str,str,str,str,str,str,str
0,"""ENSG00000223972.5""","""ENST00000456328.2""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""processed_transcript""","""DDX11L1-202""",1,"""ENSE00002234944.1""",2,"""1""","""HGNC:37102""","""basic""","""OTTHUMG00000000961.2""","""OTTHUMT00000362751.1""""",null,null,null
1,"""ENSG00000223972.5""",null,"""transcribed_unprocessed_pseudo…","""DDX11L1""",null,null,null,null,2,null,"""HGNC:37102""",null,"""OTTHUMG00000000961.2""""",null,null,null,null
2,"""ENSG00000223972.5""","""ENST00000456328.2""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""processed_transcript""","""DDX11L1-202""",null,null,2,"""1""","""HGNC:37102""","""basic""","""OTTHUMG00000000961.2""","""OTTHUMT00000362751.1""""",null,null,null
3,"""ENSG00000223972.5""","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""",1,"""ENSE00001948541.1""",2,"""NA""","""HGNC:37102""","""basic, Ensembl_canonical""","""OTTHUMG00000000961.2""","""OTTHUMT00000002844.2""""","""PGO:0000005, PGO:0000019""",null,null
4,"""ENSG00000223972.5""","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""",null,null,2,"""NA""","""HGNC:37102""","""basic, Ensembl_canonical""","""OTTHUMG00000000961.2""","""OTTHUMT00000002844.2""""","""PGO:0000005, PGO:0000019""",null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3283857,"""ENSG00000227159.8_PAR_Y""","""ENST00000507418.6_PAR_Y""","""unprocessed_pseudogene""","""DDX11L16""","""unprocessed_pseudogene""","""DDX11L16-201""",null,null,2,"""NA""","""HGNC:37115""","""basic, Ensembl_canonical, PAR""","""OTTHUMG00000022678.1""","""OTTHUMT00000058841.1""""","""PGO:0000005""",null,null
3283858,"""ENSG00000227159.8_PAR_Y""","""ENST00000507418.6_PAR_Y""","""unprocessed_pseudogene""","""DDX11L16""","""unprocessed_pseudogene""","""DDX11L16-201""",4,"""ENSE00002036959.1""",2,"""NA""","""HGNC:37115""","""basic, Ensembl_canonical, PAR""","""OTTHUMG00000022678.1""","""OTTHUMT00000058841.1""""","""PGO:0000005""",null,null
3283859,"""ENSG00000227159.8_PAR_Y""","""ENST00000507418.6_PAR_Y""","""unprocessed_pseudogene""","""DDX11L16""","""unprocessed_pseudogene""","""DDX11L16-201""",3,"""ENSE00002021169.1""",2,"""NA""","""HGNC:37115""","""basic, Ensembl_canonical, PAR""","""OTTHUMG00000022678.1""","""OTTHUMT00000058841.1""""","""PGO:0000005""",null,null


In [5]:
gtf_exp = gtf_pl.join(atts, on='row_nr')
gtf_exp

row_nr,Chromosome,source,Feature,Start,End,score,Strand,frame,attributes,gene_id,transcript_id,gene_type,gene_name,transcript_type,transcript_name,exon_number,exon_id,level,transcript_support_level,hgnc_id,tag,havana_gene,havana_transcript,ont,protein_id,ccdsid
u64,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,i32,str,i32,str,str,str,str,str,str,str,str
0,"""chr1""","""HAVANA""","""exon""",11869,12227,""".""","""+""",""".""","""gene_id ""ENSG00000223972.5""; t…","""ENSG00000223972.5""","""ENST00000456328.2""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""processed_transcript""","""DDX11L1-202""",1,"""ENSE00002234944.1""",2,"""1""","""HGNC:37102""","""basic""","""OTTHUMG00000000961.2""","""OTTHUMT00000362751.1""""",null,null,null
1,"""chr1""","""HAVANA""","""gene""",11869,14409,""".""","""+""",""".""","""gene_id ""ENSG00000223972.5""; g…","""ENSG00000223972.5""",null,"""transcribed_unprocessed_pseudo…","""DDX11L1""",null,null,null,null,2,null,"""HGNC:37102""",null,"""OTTHUMG00000000961.2""""",null,null,null,null
2,"""chr1""","""HAVANA""","""transcript""",11869,14409,""".""","""+""",""".""","""gene_id ""ENSG00000223972.5""; t…","""ENSG00000223972.5""","""ENST00000456328.2""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""processed_transcript""","""DDX11L1-202""",null,null,2,"""1""","""HGNC:37102""","""basic""","""OTTHUMG00000000961.2""","""OTTHUMT00000362751.1""""",null,null,null
3,"""chr1""","""HAVANA""","""exon""",12010,12057,""".""","""+""",""".""","""gene_id ""ENSG00000223972.5""; t…","""ENSG00000223972.5""","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""",1,"""ENSE00001948541.1""",2,"""NA""","""HGNC:37102""","""basic, Ensembl_canonical""","""OTTHUMG00000000961.2""","""OTTHUMT00000002844.2""""","""PGO:0000005, PGO:0000019""",null,null
4,"""chr1""","""HAVANA""","""transcript""",12010,13670,""".""","""+""",""".""","""gene_id ""ENSG00000223972.5""; t…","""ENSG00000223972.5""","""ENST00000450305.2""","""transcribed_unprocessed_pseudo…","""DDX11L1""","""transcribed_unprocessed_pseudo…","""DDX11L1-201""",null,null,2,"""NA""","""HGNC:37102""","""basic, Ensembl_canonical""","""OTTHUMG00000000961.2""","""OTTHUMT00000002844.2""""","""PGO:0000005, PGO:0000019""",null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3283857,"""chrY""","""HAVANA""","""transcript""",57212184,57214397,""".""","""-""",""".""","""gene_id ""ENSG00000227159.8_PAR…","""ENSG00000227159.8_PAR_Y""","""ENST00000507418.6_PAR_Y""","""unprocessed_pseudogene""","""DDX11L16""","""unprocessed_pseudogene""","""DDX11L16-201""",null,null,2,"""NA""","""HGNC:37115""","""basic, Ensembl_canonical, PAR""","""OTTHUMG00000022678.1""","""OTTHUMT00000058841.1""""","""PGO:0000005""",null,null
3283858,"""chrY""","""HAVANA""","""exon""",57213204,57213357,""".""","""-""",""".""","""gene_id ""ENSG00000227159.8_PAR…","""ENSG00000227159.8_PAR_Y""","""ENST00000507418.6_PAR_Y""","""unprocessed_pseudogene""","""DDX11L16""","""unprocessed_pseudogene""","""DDX11L16-201""",4,"""ENSE00002036959.1""",2,"""NA""","""HGNC:37115""","""basic, Ensembl_canonical, PAR""","""OTTHUMG00000022678.1""","""OTTHUMT00000058841.1""""","""PGO:0000005""",null,null
3283859,"""chrY""","""HAVANA""","""exon""",57213526,57213602,""".""","""-""",""".""","""gene_id ""ENSG00000227159.8_PAR…","""ENSG00000227159.8_PAR_Y""","""ENST00000507418.6_PAR_Y""","""unprocessed_pseudogene""","""DDX11L16""","""unprocessed_pseudogene""","""DDX11L16-201""",3,"""ENSE00002021169.1""",2,"""NA""","""HGNC:37115""","""basic, Ensembl_canonical, PAR""","""OTTHUMG00000022678.1""","""OTTHUMT00000058841.1""""","""PGO:0000005""",null,null


In [6]:
a = gtf_exp['gene_type'].value_counts(sort=True)
a

gene_type,count
str,u64
"""protein_coding""",2909967
"""lncRNA""",258393
"""processed_pseudogene""",32087
"""transcribed_unprocessed_pseudo…",27286
"""unprocessed_pseudogene""",13080
…,…
"""translated_processed_pseudogen…",8
"""Mt_rRNA""",6
"""IG_pseudogene""",3


In [10]:
mane_exons = (
    gtf_exp
    .filter(
        (pl.col('gene_type') == 'protein_coding') |
        # (pl.col('gene_type').str.contains('pseudogene')) |
        (pl.col('Feature') == 'CDS')
    )
    .filter(pl.col('Feature') != 'transcript')
    .filter(pl.col('tag').str.contains('MANE_Select'))
    # .filter(pl.col('tag').str.contains('MANE_Select') | pl.col('tag').str.contains('Ensembl_canonical'))
)

mane_exons

row_nr,Chromosome,source,Feature,Start,End,score,Strand,frame,attributes,gene_id,transcript_id,gene_type,gene_name,transcript_type,transcript_name,exon_number,exon_id,level,transcript_support_level,hgnc_id,tag,havana_gene,havana_transcript,ont,protein_id,ccdsid
u64,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,i32,str,i32,str,str,str,str,str,str,str,str
57,"""chr1""","""HAVANA""","""exon""",65419,65433,""".""","""+""",""".""","""gene_id ""ENSG00000186092.7""; t…","""ENSG00000186092.7""","""ENST00000641515.2""","""protein_coding""","""OR4F5""","""protein_coding""","""OR4F5-201""",1,"""ENSE00003812156.1""",2,null,"""HGNC:14825""","""RNA_Seq_supported_partial, bas…","""OTTHUMG00000001094.4""","""OTTHUMT00000003223.4""""",null,"""ENSP00000493376.2""",null
60,"""chr1""","""HAVANA""","""UTR""",65419,65433,""".""","""+""",""".""","""gene_id ""ENSG00000186092.7""; t…","""ENSG00000186092.7""","""ENST00000641515.2""","""protein_coding""","""OR4F5""","""protein_coding""","""OR4F5-201""",1,"""ENSE00003812156.1""",2,null,"""HGNC:14825""","""RNA_Seq_supported_partial, bas…","""OTTHUMG00000001094.4""","""OTTHUMT00000003223.4""""",null,"""ENSP00000493376.2""",null
61,"""chr1""","""HAVANA""","""exon""",65520,65573,""".""","""+""",""".""","""gene_id ""ENSG00000186092.7""; t…","""ENSG00000186092.7""","""ENST00000641515.2""","""protein_coding""","""OR4F5""","""protein_coding""","""OR4F5-201""",2,"""ENSE00003813641.1""",2,null,"""HGNC:14825""","""RNA_Seq_supported_partial, bas…","""OTTHUMG00000001094.4""","""OTTHUMT00000003223.4""""",null,"""ENSP00000493376.2""",null
62,"""chr1""","""HAVANA""","""UTR""",65520,65564,""".""","""+""",""".""","""gene_id ""ENSG00000186092.7""; t…","""ENSG00000186092.7""","""ENST00000641515.2""","""protein_coding""","""OR4F5""","""protein_coding""","""OR4F5-201""",2,"""ENSE00003813641.1""",2,null,"""HGNC:14825""","""RNA_Seq_supported_partial, bas…","""OTTHUMG00000001094.4""","""OTTHUMT00000003223.4""""",null,"""ENSP00000493376.2""",null
63,"""chr1""","""HAVANA""","""CDS""",65565,65573,""".""","""+""","""0""","""gene_id ""ENSG00000186092.7""; t…","""ENSG00000186092.7""","""ENST00000641515.2""","""protein_coding""","""OR4F5""","""protein_coding""","""OR4F5-201""",2,"""ENSE00003813641.1""",2,null,"""HGNC:14825""","""RNA_Seq_supported_partial, bas…","""OTTHUMG00000001094.4""","""OTTHUMT00000003223.4""""",null,"""ENSP00000493376.2""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3283737,"""chrY""","""HAVANA""","""exon""",57194043,57194127,""".""","""+""",""".""","""gene_id ""ENSG00000124334.18_PA…","""ENSG00000124334.18_PAR_Y""","""ENST00000244174.11_PAR_Y""","""protein_coding""","""IL9R""","""protein_coding""","""IL9R-201""",8,"""ENSE00003659134.1""",2,"""1""","""HGNC:6030""","""basic, Ensembl_canonical, MANE…","""OTTHUMG00000022720.1""","""OTTHUMT00000058981.1""""",null,"""ENSP00000244174.5""","""CCDS14771.4"""
3283742,"""chrY""","""HAVANA""","""CDS""",57196336,57196926,""".""","""+""","""0""","""gene_id ""ENSG00000124334.18_PA…","""ENSG00000124334.18_PAR_Y""","""ENST00000244174.11_PAR_Y""","""protein_coding""","""IL9R""","""protein_coding""","""IL9R-201""",9,"""ENSE00001614034.3""",2,"""1""","""HGNC:6030""","""basic, Ensembl_canonical, MANE…","""OTTHUMG00000022720.1""","""OTTHUMT00000058981.1""""",null,"""ENSP00000244174.5""","""CCDS14771.4"""
3283743,"""chrY""","""HAVANA""","""exon""",57196336,57197337,""".""","""+""",""".""","""gene_id ""ENSG00000124334.18_PA…","""ENSG00000124334.18_PAR_Y""","""ENST00000244174.11_PAR_Y""","""protein_coding""","""IL9R""","""protein_coding""","""IL9R-201""",9,"""ENSE00001614034.3""",2,"""1""","""HGNC:6030""","""basic, Ensembl_canonical, MANE…","""OTTHUMG00000022720.1""","""OTTHUMT00000058981.1""""",null,"""ENSP00000244174.5""","""CCDS14771.4"""


In [11]:
# One transcript per gene
mane_exons[['gene_id', 'gene_name', 'transcript_id']].unique()['transcript_id'].value_counts(sort=True)

transcript_id,count
str,u64
"""ENST00000561421.6""",1
"""ENST00000342136.9""",1
"""ENST00000397062.8""",1
"""ENST00000614987.5""",1
"""ENST00000683969.1""",1
…,…
"""ENST00000444304.3""",1
"""ENST00000263736.5""",1
"""ENST00000184956.11""",1


In [12]:
a = mane_exons['gene_type'].value_counts(sort=True)
a

gene_type,count
str,u64
"""protein_coding""",467200


In [13]:
b = mane_exons['Feature'].value_counts(sort=True)
b

Feature,count
str,u64
"""exon""",196563
"""CDS""",186748
"""UTR""",46605
"""start_codon""",18631
"""stop_codon""",18619
"""Selenocysteine""",34


In [14]:
non_mane_exons = (
    gtf_exp
    .filter(
        (pl.col('gene_type') == 'protein_coding') |
        # (pl.col('gene_type').str.contains('pseudogene')) |
        (pl.col('Feature') == 'CDS')
    )
    .filter(pl.col('Feature') != 'transcript')
    .filter(~pl.col('tag').str.contains('MANE_Select'))
)

non_mane_exons

row_nr,Chromosome,source,Feature,Start,End,score,Strand,frame,attributes,gene_id,transcript_id,gene_type,gene_name,transcript_type,transcript_name,exon_number,exon_id,level,transcript_support_level,hgnc_id,tag,havana_gene,havana_transcript,ont,protein_id,ccdsid
u64,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,i32,str,i32,str,str,str,str,str,str,str,str
941,"""chr1""","""HAVANA""","""exon""",923923,924948,""".""","""+""",""".""","""gene_id ""ENSG00000187634.13""; …","""ENSG00000187634.13""","""ENST00000618323.5""","""protein_coding""","""SAMD11""","""protein_coding""","""SAMD11-213""",1,"""ENSE00001637883.3""",2,"""5""","""HGNC:28706""","""CAGE_supported_TSS, basic, app…","""OTTHUMG00000040719.11""""",null,null,"""ENSP00000480678.2""",null
946,"""chr1""","""HAVANA""","""UTR""",923923,924431,""".""","""+""",""".""","""gene_id ""ENSG00000187634.13""; …","""ENSG00000187634.13""","""ENST00000618323.5""","""protein_coding""","""SAMD11""","""protein_coding""","""SAMD11-213""",1,"""ENSE00001637883.3""",2,"""5""","""HGNC:28706""","""CAGE_supported_TSS, basic, app…","""OTTHUMG00000040719.11""""",null,null,"""ENSP00000480678.2""",null
948,"""chr1""","""HAVANA""","""CDS""",924432,924948,""".""","""+""","""0""","""gene_id ""ENSG00000187634.13""; …","""ENSG00000187634.13""","""ENST00000618323.5""","""protein_coding""","""SAMD11""","""protein_coding""","""SAMD11-213""",1,"""ENSE00001637883.3""",2,"""5""","""HGNC:28706""","""CAGE_supported_TSS, basic, app…","""OTTHUMG00000040719.11""""",null,null,"""ENSP00000480678.2""",null
950,"""chr1""","""HAVANA""","""start_codon""",924432,924434,""".""","""+""","""0""","""gene_id ""ENSG00000187634.13""; …","""ENSG00000187634.13""","""ENST00000618323.5""","""protein_coding""","""SAMD11""","""protein_coding""","""SAMD11-213""",1,"""ENSE00001637883.3""",2,"""5""","""HGNC:28706""","""CAGE_supported_TSS, basic, app…","""OTTHUMG00000040719.11""""",null,null,"""ENSP00000480678.2""",null
951,"""chr1""","""HAVANA""","""exon""",925150,925189,""".""","""+""",""".""","""gene_id ""ENSG00000187634.13""; …","""ENSG00000187634.13""","""ENST00000437963.5""","""protein_coding""","""SAMD11""","""protein_coding""","""SAMD11-203""",1,"""ENSE00001481182.1""",2,"""5""","""HGNC:28706""","""alternative_5_UTR, mRNA_end_NF…","""OTTHUMG00000040719.11""","""OTTHUMT00000097862.5""""",null,"""ENSP00000393181.1""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3283850,"""chrY""","""HAVANA""","""exon""",57211761,57212186,""".""","""+""",""".""","""gene_id ""ENSG00000182484.15_PA…","""ENSG00000182484.15_PAR_Y""","""ENST00000340131.12_PAR_Y""","""protein_coding""","""WASH6P""","""retained_intron""","""WASH6P-201""",8,"""ENSE00001956598.1""",2,"""1""","""HGNC:31685""","""PAR""","""OTTHUMG00000022677.5""","""OTTHUMT00000058837.1""""",null,null,null
3283851,"""chrY""","""HAVANA""","""exon""",57211761,57212186,""".""","""+""",""".""","""gene_id ""ENSG00000182484.15_PA…","""ENSG00000182484.15_PAR_Y""","""ENST00000492963.6_PAR_Y""","""protein_coding""","""WASH6P""","""retained_intron""","""WASH6P-213""",6,"""ENSE00001956598.1""",2,"""5""","""HGNC:31685""","""PAR""","""OTTHUMG00000022677.5""","""OTTHUMT00000058839.1""""",null,null,null
3283852,"""chrY""","""HAVANA""","""exon""",57211761,57212230,""".""","""+""",""".""","""gene_id ""ENSG00000182484.15_PA…","""ENSG00000182484.15_PAR_Y""","""ENST00000460206.6_PAR_Y""","""protein_coding""","""WASH6P""","""retained_intron""","""WASH6P-203""",8,"""ENSE00001856109.1""",2,"""5""","""HGNC:31685""","""non_canonical_TEC, not_best_in…","""OTTHUMG00000022677.5""","""OTTHUMT00000058838.1""""",null,null,null


In [15]:
non_mane_exons[['gene_id', 'gene_name', 'transcript_id']].unique()['transcript_id'].value_counts(sort=True)

transcript_id,count
str,u64
null,6695
"""ENST00000392884.2""",1
"""ENST00000457869.1""",1
"""ENST00000534943.5""",1
"""ENST00000299613.10""",1
…,…
"""ENST00000688981.1""",1
"""ENST00000295868.6""",1
"""ENST00000632951.1""",1


In [17]:
non_mane_cds = non_mane_exons.filter(pl.col('Feature') == 'CDS')
non_mane_cds

row_nr,Chromosome,source,Feature,Start,End,score,Strand,frame,attributes,gene_id,transcript_id,gene_type,gene_name,transcript_type,transcript_name,exon_number,exon_id,level,transcript_support_level,hgnc_id,tag,havana_gene,havana_transcript,ont,protein_id,ccdsid
u64,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,i32,str,i32,str,str,str,str,str,str,str,str
948,"""chr1""","""HAVANA""","""CDS""",924432,924948,""".""","""+""","""0""","""gene_id ""ENSG00000187634.13""; …","""ENSG00000187634.13""","""ENST00000618323.5""","""protein_coding""","""SAMD11""","""protein_coding""","""SAMD11-213""",1,"""ENSE00001637883.3""",2,"""5""","""HGNC:28706""","""CAGE_supported_TSS, basic, app…","""OTTHUMG00000040719.11""""",null,null,"""ENSP00000480678.2""",null
958,"""chr1""","""HAVANA""","""CDS""",925922,926013,""".""","""+""","""2""","""gene_id ""ENSG00000187634.13""; …","""ENSG00000187634.13""","""ENST00000618323.5""","""protein_coding""","""SAMD11""","""protein_coding""","""SAMD11-213""",2,"""ENSE00003794726.1""",2,"""5""","""HGNC:28706""","""CAGE_supported_TSS, basic, app…","""OTTHUMG00000040719.11""""",null,null,"""ENSP00000480678.2""",null
965,"""chr1""","""HAVANA""","""CDS""",925942,926013,""".""","""+""","""0""","""gene_id ""ENSG00000187634.13""; …","""ENSG00000187634.13""","""ENST00000342066.8""","""protein_coding""","""SAMD11""","""protein_coding""","""SAMD11-202""",2,"""ENSE00003902988.1""",2,"""5""","""HGNC:28706""","""basic, CCDS""","""OTTHUMG00000040719.11""","""OTTHUMT00000276866.3""""",null,"""ENSP00000342313.3""","""CCDS2.2"""
966,"""chr1""","""HAVANA""","""CDS""",925942,926013,""".""","""+""","""0""","""gene_id ""ENSG00000187634.13""; …","""ENSG00000187634.13""","""ENST00000437963.5""","""protein_coding""","""SAMD11""","""protein_coding""","""SAMD11-203""",2,"""ENSE00003902988.1""",2,"""5""","""HGNC:28706""","""alternative_5_UTR, mRNA_end_NF…","""OTTHUMG00000040719.11""","""OTTHUMT00000097862.5""""",null,"""ENSP00000393181.1""",null
967,"""chr1""","""HAVANA""","""CDS""",925942,926013,""".""","""+""","""0""","""gene_id ""ENSG00000187634.13""; …","""ENSG00000187634.13""","""ENST00000616125.5""","""protein_coding""","""SAMD11""","""protein_coding""","""SAMD11-210""",1,"""ENSE00003903077.2""",2,"""5""","""HGNC:28706""","""mRNA_start_NF, cds_start_NF""","""OTTHUMG00000040719.11""""",null,null,"""ENSP00000484643.1""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3283809,"""chrY""","""HAVANA""","""CDS""",57209532,57209733,""".""","""+""","""0""","""gene_id ""ENSG00000182484.15_PA…","""ENSG00000182484.15_PAR_Y""","""ENST00000359512.8_PAR_Y""","""protein_coding""","""WASH6P""","""protein_coding""","""WASH6P-202""",6,"""ENSE00003910082.1""",2,"""NA""","""HGNC:31685""","""basic, Ensembl_canonical, appr…","""OTTHUMG00000022677.5""""",null,null,"""ENSP00000504557.1""",null
3283819,"""chrY""","""HAVANA""","""CDS""",57209822,57209980,""".""","""+""","""2""","""gene_id ""ENSG00000182484.15_PA…","""ENSG00000182484.15_PAR_Y""","""ENST00000359512.8_PAR_Y""","""protein_coding""","""WASH6P""","""protein_coding""","""WASH6P-202""",7,"""ENSE00003908490.1""",2,"""NA""","""HGNC:31685""","""basic, Ensembl_canonical, appr…","""OTTHUMG00000022677.5""""",null,null,"""ENSP00000504557.1""",null
3283835,"""chrY""","""HAVANA""","""CDS""",57210640,57210792,""".""","""+""","""2""","""gene_id ""ENSG00000182484.15_PA…","""ENSG00000182484.15_PAR_Y""","""ENST00000359512.8_PAR_Y""","""protein_coding""","""WASH6P""","""protein_coding""","""WASH6P-202""",8,"""ENSE00003906775.1""",2,"""NA""","""HGNC:31685""","""basic, Ensembl_canonical, appr…","""OTTHUMG00000022677.5""""",null,null,"""ENSP00000504557.1""",null


In [18]:
non_mane_exonic = non_mane_exons.filter(pl.col('Feature') == 'exon')
non_mane_exonic

row_nr,Chromosome,source,Feature,Start,End,score,Strand,frame,attributes,gene_id,transcript_id,gene_type,gene_name,transcript_type,transcript_name,exon_number,exon_id,level,transcript_support_level,hgnc_id,tag,havana_gene,havana_transcript,ont,protein_id,ccdsid
u64,str,str,str,i64,i64,str,str,str,str,str,str,str,str,str,str,i32,str,i32,str,str,str,str,str,str,str,str
941,"""chr1""","""HAVANA""","""exon""",923923,924948,""".""","""+""",""".""","""gene_id ""ENSG00000187634.13""; …","""ENSG00000187634.13""","""ENST00000618323.5""","""protein_coding""","""SAMD11""","""protein_coding""","""SAMD11-213""",1,"""ENSE00001637883.3""",2,"""5""","""HGNC:28706""","""CAGE_supported_TSS, basic, app…","""OTTHUMG00000040719.11""""",null,null,"""ENSP00000480678.2""",null
951,"""chr1""","""HAVANA""","""exon""",925150,925189,""".""","""+""",""".""","""gene_id ""ENSG00000187634.13""; …","""ENSG00000187634.13""","""ENST00000437963.5""","""protein_coding""","""SAMD11""","""protein_coding""","""SAMD11-203""",1,"""ENSE00001481182.1""",2,"""5""","""HGNC:28706""","""alternative_5_UTR, mRNA_end_NF…","""OTTHUMG00000040719.11""","""OTTHUMT00000097862.5""""",null,"""ENSP00000393181.1""",null
954,"""chr1""","""HAVANA""","""exon""",925731,925800,""".""","""+""",""".""","""gene_id ""ENSG00000187634.13""; …","""ENSG00000187634.13""","""ENST00000342066.8""","""protein_coding""","""SAMD11""","""protein_coding""","""SAMD11-202""",1,"""ENSE00001864899.2""",2,"""5""","""HGNC:28706""","""basic, CCDS""","""OTTHUMG00000040719.11""","""OTTHUMT00000276866.3""""",null,"""ENSP00000342313.3""","""CCDS2.2"""
959,"""chr1""","""HAVANA""","""exon""",925922,926013,""".""","""+""",""".""","""gene_id ""ENSG00000187634.13""; …","""ENSG00000187634.13""","""ENST00000342066.8""","""protein_coding""","""SAMD11""","""protein_coding""","""SAMD11-202""",2,"""ENSE00003902988.1""",2,"""5""","""HGNC:28706""","""basic, CCDS""","""OTTHUMG00000040719.11""","""OTTHUMT00000276866.3""""",null,"""ENSP00000342313.3""","""CCDS2.2"""
960,"""chr1""","""HAVANA""","""exon""",925922,926013,""".""","""+""",""".""","""gene_id ""ENSG00000187634.13""; …","""ENSG00000187634.13""","""ENST00000437963.5""","""protein_coding""","""SAMD11""","""protein_coding""","""SAMD11-203""",2,"""ENSE00003902988.1""",2,"""5""","""HGNC:28706""","""alternative_5_UTR, mRNA_end_NF…","""OTTHUMG00000040719.11""","""OTTHUMT00000097862.5""""",null,"""ENSP00000393181.1""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
3283848,"""chrY""","""HAVANA""","""exon""",57211761,57212074,""".""","""+""",""".""","""gene_id ""ENSG00000182484.15_PA…","""ENSG00000182484.15_PAR_Y""","""ENST00000464205.6_PAR_Y""","""protein_coding""","""WASH6P""","""processed_transcript""","""WASH6P-205""",3,"""ENSE00001531705.3""",2,"""2""","""HGNC:31685""","""PAR""","""OTTHUMG00000022677.5""","""OTTHUMT00000058835.1""""",null,null,null
3283849,"""chrY""","""HAVANA""","""exon""",57211761,57212074,""".""","""+""",""".""","""gene_id ""ENSG00000182484.15_PA…","""ENSG00000182484.15_PAR_Y""","""ENST00000483286.6_PAR_Y""","""protein_coding""","""WASH6P""","""retained_intron""","""WASH6P-211""",3,"""ENSE00001531705.3""",2,"""1""","""HGNC:31685""","""PAR""","""OTTHUMG00000022677.5""","""OTTHUMT00000058834.1""""",null,null,null
3283850,"""chrY""","""HAVANA""","""exon""",57211761,57212186,""".""","""+""",""".""","""gene_id ""ENSG00000182484.15_PA…","""ENSG00000182484.15_PAR_Y""","""ENST00000340131.12_PAR_Y""","""protein_coding""","""WASH6P""","""retained_intron""","""WASH6P-201""",8,"""ENSE00001956598.1""",2,"""1""","""HGNC:31685""","""PAR""","""OTTHUMG00000022677.5""","""OTTHUMT00000058837.1""""",null,null,null


### Convert regions to pyranges

In [19]:
mane_exons_pr = pr.PyRanges(mane_exons.filter(pl.col('Feature') == 'exon').to_pandas()).merge()
mane_exons_pr

,Chromosome,Start,End,Strand
0,chr1,65419,65433,+
1,chr1,65520,65573,+
2,chr1,69037,71585,+
3,chr1,923923,924948,+
4,chr1,925922,926013,+
...,...,...,...,...
196312,chrY,25041769,25041886,-
196313,chrY,25043946,25044064,-
196314,chrY,25048474,25048610,-
196315,chrY,25051676,25051798,-


In [20]:
mane_cds_pr = pr.PyRanges(mane_exons.filter(pl.col('Feature') == 'CDS').to_pandas()).merge()
mane_utr_pr = pr.PyRanges(mane_exons.filter(pl.col('Feature') == 'UTR').to_pandas()).merge()

mane_cds_pr

,Chromosome,Start,End,Strand
0,chr1,65565,65573,+
1,chr1,69037,70005,+
2,chr1,924432,924948,+
3,chr1,925922,926013,+
4,chr1,930155,930336,+
...,...,...,...,...
186591,chrY,24813183,24813185,-
186592,chrY,25038101,25038116,-
186593,chrY,25038809,25038914,-
186594,chrY,25041769,25041886,-


In [21]:
non_mane_exons_pr = pr.PyRanges(non_mane_exons.filter(pl.col('Feature') == 'exon').to_pandas()).merge()
non_mane_exons_pr = non_mane_exons_pr.subtract(mane_exons_pr)
non_mane_exons_pr

,Chromosome,Start,End,Strand
0,chr1,925150,925189,+
1,chr1,925731,925800,+
2,chr1,939272,939275,+
3,chr1,939412,939460,+
4,chr1,943808,943908,+
...,...,...,...,...
70219,chrY,23694459,23694579,-
70220,chrY,24045229,24045676,-
70221,chrY,24813393,24813492,-
70222,chrY,25052151,25052219,-


In [22]:
non_mane_cds_pr = pr.PyRanges(non_mane_exons.filter(pl.col('Feature') == 'CDS').to_pandas()).merge()
non_mane_cds_pr = non_mane_cds_pr.subtract(mane_cds_pr)

non_mane_utr_pr = pr.PyRanges(non_mane_exons.filter(pl.col('Feature') == 'UTR').to_pandas()).merge()
non_mane_utr_pr = non_mane_utr_pr.subtract(mane_utr_pr)

non_mane_cds_pr

,Chromosome,Start,End,Strand
0,chr1,939272,939275,+
1,chr1,939412,939460,+
2,chr1,943808,943908,+
3,chr1,963032,963109,+
4,chr1,970758,970879,+
...,...,...,...,...
33713,chrY,19723341,19723433,-
33714,chrY,23682010,23682265,-
33715,chrY,23691757,23691872,-
33716,chrY,23694459,23694527,-


## Merge with UKBBGym annotations

In [42]:
# Download annotations file

anno_dir = '/s/project/deeprvat_wgs/input_data/annotations/qced_maf1e-3_loftee_olink_genes_EURunrelated/'
anno_file = "annotations_fillna_ukbgym"

!dx download project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/annotations_fillna_ukbgym.parquet -o {anno_dir}

anno_df = pl.scan_parquet(f'{anno_dir}/{anno_file}.parquet')
anno_df.head().collect()

Error: path "/s/project/deeprvat_wgs/input_data/annotations/qced_maf1e-3_lofte
e_olink_genes_EURunrelated/annotations_fillna_ukbgym.parquet" already exists
but -f/--overwrite was not set


id,chrom,pos,ref,alt,region,sift,polyphen,cadd_phred,cadd_raw,am_pathogenicity,spliceai_pred,loftee_hc,loftee_lc,relative_cds_position,next_in_frame_relative,spliceai_delta_score,pangolin_score,delta_score,absplice_dna_max,absplice2_max,five_prime_utr_variant_consequence_uaug_gained,five_prime_utr_variant_consequence_uaug_lost,five_prime_utr_variant_consequence_uframeshift,five_prime_utr_variant_consequence_ustop_gained,five_prime_utr_variant_consequence_ustop_lost,variant_length,gpn_score,promoterai,score_pai3d,abexp_abs_max,gene_length,dist_to_tss,aparent2,cdspos,cpg,dst2splice,…,motifecount_is_na,motifehipos_is_na,motifescorechng_is_na,priphcons_is_na,priphylop_is_na,relcdspos_is_na,relprotpos_is_na,relcdnapos_is_na,toverlapmotifs_is_na,targetscan_is_na,verphcons_is_na,verphylop_is_na,sc_ukb,ac_ukb,mac_ukb,vep_cds_relaxed,tss,strand,gene_length_right,gene_name,dist_to_tss_right,region_right,protein_position,amino_acids,mobi_curated_disorder_priority,mobi_lip_full,ted_domain,low_complexity_domain,not_annotated_in_encode,encode_dels,encode_ca-ctcf,encode_ca,encode_ca-h3k4me3,encode_tf,encode_ca-tf,encode_pels,encode_pls
str,str,i64,str,str,str,f32,f32,f32,f32,f32,str,i8,i8,f32,f32,f32,f32,f32,f32,f32,u8,u8,u8,u8,u8,u32,f32,f32,f32,f32,i64,i64,f32,f32,f32,f32,…,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,u64,i64,i64,bool,i64,cat,i64,str,i64,str,str,str,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool
"""chr2:26136528:A:G""","""chr2""",26136528,"""A""","""G""","""ENSG00000084733""",1.0,0.0,0.83,-0.065577,0.0,"""RAB10|0.00|0.00|0.00|0.00|-30|…",0,0,0.0,0.0,0.0,0.01,0.0,0.00057,0.000034,0,0,0,0,0,1,-2.15,0.0,0.0,0.005917,103372,102445,0.0,0.0,0.013,0.0,…,1,1,1,0,0,1,1,0,1,1,0,0,2,2,2,false,26034083,"""+""",103372,"""RAB10""",102445,"""ENSG00000084733""",null,null,false,false,false,false,false,true,false,false,false,false,false,false,false
"""chr2:233771145:C:T""","""chr2""",233771145,"""C""","""T""","""ENSG00000241119""",1.0,0.0,1.374,0.065026,0.0,"""UGT1A9|0.00|0.00|0.00|0.00|-47…",0,0,0.0,0.0,0.0,0.0,0.0,0.001,0.000033,0,0,0,0,0,1,-0.93,0.0,0.0,0.019552,101404,99248,0.0,0.0,0.013,0.0,…,1,1,1,0,0,1,1,1,1,1,0,0,3,3,3,false,233671897,"""+""",101404,"""UGT1A9""",99248,"""ENSG00000167165""",null,null,false,false,false,false,true,false,false,false,false,false,false,false,false
"""chr12:101117325:T:G""","""chr12""",101117325,"""T""","""G""","""ENSG00000151572""",1.0,0.0,7.157,0.675173,0.0,"""ANO4|0.06|0.00|0.00|0.00|-8|45…",0,0,0.0,0.0,0.06,0.0,0.0,0.0,0.0,0,0,0,0,0,1,-1.51,0.0,0.0,0.006774,411117,399800,0.0,0.0,0.013,0.0,…,1,1,1,0,0,1,1,1,0,1,0,0,2,2,2,false,100717525,"""+""",411117,"""ANO4""",399800,"""ENSG00000151572""",null,null,false,false,false,false,true,false,false,false,false,false,false,false,false
"""chr10:25235985:T:G""","""chr10""",25235985,"""T""","""G""","""ENSG00000151025""",1.0,0.0,0.297,-0.324639,0.0,"""GPR158|0.00|0.00|0.00|0.00|-35…",0,0,0.0,0.0,0.0,0.0,0.0,0.001,0.000033,0,0,0,0,0,1,-1.7,0.0,0.0,0.006286,427429,61184,0.0,0.0,0.04,0.0,…,1,1,1,0,0,1,1,1,1,1,0,0,1,1,1,false,25174801,"""+""",427429,"""GPR158""",61184,"""ENSG00000151025""",null,null,false,false,false,false,true,false,false,false,false,false,false,false,false
"""chr7:80447426:C:CA""","""chr7""",80447426,"""C""","""CA""","""ENSG00000135218""",1.0,0.0,1.117,0.010904,0.0,"""CD36|0.00|0.00|0.00|0.00|-29|-…",0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,2,-1.41,0.0,0.0,0.0,309704,77852,0.0,0.0,0.0,0.0,…,1,1,1,1,1,1,1,1,1,1,1,1,2,2,2,false,80369574,"""+""",309704,"""CD36""",77852,"""ENSG00000135218""",null,null,false,false,false,false,true,false,false,false,false,false,false,false,false


In [29]:
anno = (
    anno_df
    .with_columns(
        Chromosome = pl.col('chrom'),
        Start = pl.col('pos') - 1,
        End = pl.col('pos') - 1 + pl.col('ref').str.len_chars(),
        gene = pl.col('region')
    )
    .select(['id', 'Chromosome', 'Start', 'End', 'ref', 'alt', 'gene'])
    .collect(engine='streaming')
)

anno_pr = pr.PyRanges(anno.to_pandas())
anno_pr

,id,Chromosome,Start,End,ref,alt,gene
0,chr1:39747598:G:A,chr1,39747597,39747598,G,A,ENSG00000084072
1,chr1:183587665:G:A,chr1,183587664,183587665,G,A,ENSG00000116701
2,chr1:231540122:C:CA,chr1,231540121,231540122,C,CA,ENSG00000116918
3,chr1:14401409:C:A,chr1,14401408,14401409,C,A,ENSG00000189337
4,chr1:28979146:A:T,chr1,28979145,28979146,A,T,ENSG00000159023
...,...,...,...,...,...,...,...
77094931,chr22:42880974:G:A,chr22,42880973,42880974,G,A,ENSG00000100266
77094932,chr22:49768435:C:T,chr22,49768434,49768435,C,T,ENSG00000100425
77094933,chr22:41397712:A:G,chr22,41397711,41397712,A,G,ENSG00000167074
77094934,chr22:20115089:TCA:T,chr22,20115088,20115091,TCA,T,ENSG00000099901


In [30]:
mane_exons_pr_vars = anno_pr.overlap(mane_exons_pr)#.df['gene'].value_counts()
mane_cds_pr_vars = anno_pr.overlap(mane_cds_pr)
mane_utr_pr_vars = anno_pr.overlap(mane_utr_pr)

mane_cds_pr_vars

,id,Chromosome,Start,End,ref,alt,gene
0,chr1:185734879:G:A,chr1,185734878,185734879,G,A,ENSG00000143341
1,chr1:154176167:T:C,chr1,154176166,154176167,T,C,ENSG00000143549
2,chr1:236835683:C:G,chr1,236835682,236835683,C,G,ENSG00000116984
3,chr1:202752969:G:A,chr1,202752968,202752969,G,A,ENSG00000117139
4,chr1:22005752:G:C,chr1,22005751,22005752,G,C,ENSG00000142789
...,...,...,...,...,...,...,...
2259799,chr22:41387392:G:A,chr22,41387391,41387392,G,A,ENSG00000167074
2259800,chr22:50521562:A:G,chr22,50521561,50521562,A,G,ENSG00000025708
2259801,chr22:41117603:G:T,chr22,41117602,41117603,G,T,ENSG00000100393
2259802,chr22:50455525:A:G,chr22,50455524,50455525,A,G,ENSG00000100241


In [31]:
# Convert to polars and add MANE / non-MANE flags

non_mane_exons_pr_vars = anno_pr.overlap(non_mane_exons_pr)
non_mane_cds_pr_vars = anno_pr.overlap(non_mane_cds_pr)
non_mane_utr_pr_vars = anno_pr.overlap(non_mane_utr_pr)

non_mane_cds_pr_vars

,id,Chromosome,Start,End,ref,alt,gene
0,chr1:186375156:C:T,chr1,186375155,186375156,C,T,ENSG00000047410
1,chr1:21868891:C:T,chr1,21868890,21868891,C,T,ENSG00000142798
2,chr1:205073509:G:A,chr1,205073508,205073509,G,A,ENSG00000184144
3,chr1:109622212:TG:T,chr1,109622211,109622213,TG,T,ENSG00000134183
4,chr1:28518674:G:C,chr1,28518673,28518674,G,C,ENSG00000180198
...,...,...,...,...,...,...,...
196808,chr22:41994741:C:T,chr22,41994740,41994741,C,T,ENSG00000100167
196809,chr22:20126484:G:A,chr22,20126483,20126484,G,A,ENSG00000099901
196810,chr22:37142564:G:A,chr22,37142563,37142564,G,A,ENSG00000100385
196811,chr22:44679837:G:A,chr22,44679836,44679837,G,A,ENSG00000186654


In [32]:
# Convert to polars and add MANE / non-MANE flags

mane_exons_pl = pl.from_pandas(mane_exons_pr_vars.df).with_columns(MANE_exonic = True).select(['id', 'gene', 'MANE_exonic'])
mane_cds_pl = pl.from_pandas(mane_cds_pr_vars.df).with_columns(MANE_CDS = True).select(['id', 'gene', 'MANE_CDS'])
mane_utr_pl = pl.from_pandas(mane_utr_pr_vars.df).with_columns(MANE_UTR = True).select(['id', 'gene', 'MANE_UTR'])

non_mane_exons_pl = pl.from_pandas(non_mane_exons_pr_vars.df).with_columns(non_MANE_exonic = True).select(['id', 'gene', 'non_MANE_exonic'])
non_mane_cds_pl = pl.from_pandas(non_mane_cds_pr_vars.df).with_columns(non_MANE_CDS = True).select(['id', 'gene', 'non_MANE_CDS'])
non_mane_utr_pl = pl.from_pandas(non_mane_utr_pr_vars.df).with_columns(non_MANE_UTR = True).select(['id', 'gene', 'non_MANE_UTR'])

non_mane_exons_pl

id,gene,non_MANE_exonic
str,str,bool
"""chr1:28539295:A:AC""","""ENSG00000180198""",true
"""chr1:75727382:T:C""","""ENSG00000117054""",true
"""chr1:206769609:G:A""","""ENSG00000136634""",true
"""chr1:169106528:A:T""","""ENSG00000143153""",true
"""chr1:207768516:A:G""","""ENSG00000117335""",true
…,…,…
"""chr22:26292317:G:A""","""ENSG00000100095""",true
"""chr22:44679837:G:A""","""ENSG00000186654""",true
"""chr22:50447874:C:G""","""ENSG00000100241""",true


In [41]:
mane_annos = (
    anno_df
    .with_columns(
        gene = pl.col('region').cast(pl.Utf8)
    )
    .select(['id', 'gene'])
    .collect(engine='streaming')

    .join(mane_exons_pl, on=['id', 'gene'], how='left')
    .join(mane_cds_pl, on=['id', 'gene'], how='left')
    .join(mane_utr_pl, on=['id', 'gene'], how='left')
    .join(non_mane_exons_pl, on=['id', 'gene'], how='left')
    .join(non_mane_cds_pl, on=['id', 'gene'], how='left')
    .join(non_mane_utr_pl, on=['id', 'gene'], how='left')

    .fill_null(False)
)

mane_annos = (
    mane_annos
    .unique()
    .rename({c:c.lower() for c in mane_annos.columns})
    .rename({'gene':'region'})
)

mane_annos

id,region,mane_exonic,mane_cds,mane_utr,non_mane_exonic,non_mane_cds,non_mane_utr
str,str,bool,bool,bool,bool,bool,bool
"""chr19:43413945:G:T""","""ENSG00000131126""",false,false,false,false,false,false
"""chr10:51708452:A:G""","""ENSG00000185532""",false,false,false,false,false,false
"""chr10:25357798:A:T""","""ENSG00000151025""",false,false,false,false,false,false
"""chr3:124298887:C:T""","""ENSG00000160145""",true,true,false,false,false,false
"""chr21:36340861:TTCTG:T""","""ENSG00000159256""",false,false,false,false,false,false
…,…,…,…,…,…,…,…
"""chr1:32794941:T:C""","""ENSG00000134684""",false,false,false,false,false,false
"""chr7:37919147:C:T""","""ENSG00000106483""",false,false,false,false,false,false
"""chr20:59019884:G:A""","""ENSG00000101162""",false,false,false,false,false,false


In [43]:
anno_df.select(pl.len()).collect()

len
u64
77094936


In [44]:
anno_df = anno_df.join(mane_annos.lazy(), on=['id', 'region'], how='left').fill_null(False)

anno_df.sink_parquet(f'{anno_dir}/{anno_file}_with_mane.parquet')

In [45]:
!dx upload {anno_dir}/{anno_file}_with_mane.parquet --path project-Gyp4fvjJg0yFZ374KvP9bGFJ:/processed_data/wgs/qced_maf1e-3_loftee_olink_genes/qced_maf1e-3_loftee_olink_genes_EURunrelated/

[===========================================================>] Uploaded 7,239,965,276 of 7,239,965,276 bytes (100%) /s/project/deeprvat_wgs/input_data/annotations/qced_maf1e-3_loftee_olink_genes_EURunrelated//annotations_fillna_ukbgym_with_mane.parquet=========>                                                  ] Uploaded 1,207,959,552 of 7,239,965,276 bytes (17%) /s/project/deeprvat_wgs/input_data/annotations/qced_maf1e-3_loftee_olink_genes_EURunrelated//annotations_fillna_ukbgym_with_mane.parquet[=========>                                                  ] Uploaded 1,224,736,768 of 7,239,965,276 bytes (17%) /s/project/deeprvat_wgs/input_data/annotations/qced_maf1e-3_loftee_olink_genes_EURunrelated//annotations_fillna_ukbgym_with_mane.parquet
ID                                file-J5y8yXjJg0yBkzQPFg1Ky0B0
Class                             file
Project                           project-Gyp4fvjJg0yFZ374KvP9bGFJ
Folder                            /processed_data/wgs/qced_maf1e-3_loftee_ol